![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)

# <h1><center>Electrical Motor Visualizer on Colab <a href="https://colab.research.google.com/github/robomotic/mujoco/blob/motors/python/examples/electrical/demo_visualizer_colab.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" width="140" align="center"/></a></center></h1>

This notebook will:

1. fetch the remote `motors` branch,
2. install the released MuJoCo wheel from GitHub Releases,
3. run the electrical visualizer demo, and
4. expose a browser port so the viewer can be opened inside Google Colab.

> No local compilation is needed. The notebook now supports both Python `3.11` and `3.12` runtimes by selecting the matching released wheel automatically.

In [ ]:
from pathlib import Path
import os
import platform
import sys

REPO_URL = 'https://github.com/robomotic/mujoco.git'
BRANCH = 'motors'
RELEASE_TAG = 'motors-wheel-v3.7.0-4'
REPO_DIR = Path('/content/mujoco')
VNC_PORT = 6080
DISPLAY_ID = ':1'

SUPPORTED_WHEELS = {
    (3, 11): 'mujoco-3.7.0-cp311-cp311-linux_x86_64.whl',
    (3, 12): 'mujoco-3.7.0-cp312-cp312-linux_x86_64.whl',
}

py_key = sys.version_info[:2]
if py_key not in SUPPORTED_WHEELS:
    raise RuntimeError(
        f'Unsupported Colab runtime {sys.version.split()[0]}. '         'Please use Python 3.11 or 3.12.'
    )

WHEEL_NAME = SUPPORTED_WHEELS[py_key]
WHEEL_URL = f'https://github.com/robomotic/mujoco/releases/download/{RELEASE_TAG}/{WHEEL_NAME}'

os.environ['REPO_URL'] = REPO_URL
os.environ['BRANCH'] = BRANCH
os.environ['REPO_DIR'] = str(REPO_DIR)
os.environ['WHEEL_URL'] = WHEEL_URL
os.environ['VNC_PORT'] = str(VNC_PORT)
os.environ['DISPLAY_ID'] = DISPLAY_ID

print(f'Python:   {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f'Repo:     {REPO_URL}')
print(f'Branch:   {BRANCH}')
print(f'Release:  {RELEASE_TAG}')
print(f'Wheel:    {WHEEL_URL}')
print(f'Port:     {VNC_PORT}')


In [ ]:
%%bash
set -euxo pipefail
apt-get update
DEBIAN_FRONTEND=noninteractive apt-get install -y \
  git libgl1-mesa-glx libglfw3 libosmesa6 mesa-utils \
  xvfb fluxbox x11vnc websockify novnc
python3 -m pip install --upgrade pip
python3 -m pip install --upgrade \
  "$WHEEL_URL" \
  glfw PyOpenGL absl-py "etils[epath]"
python3 - <<'PY'
import mujoco
print('Installed MuJoCo version:', mujoco.__version__)
print('Bundled plugin dir:', mujoco.PLUGINS_DIR)
PY

In [ ]:
%%bash
set -euxo pipefail
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone --depth=1 --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
fi
cd "$REPO_DIR"
git fetch origin "$BRANCH" --depth=1
git checkout "$BRANCH"
git pull --ff-only origin "$BRANCH"
git status --short --branch


In [ ]:
import mujoco
import mujoco.viewer
from mujoco.electrical import SingleEnvSimulation

print('Wheel import OK:', mujoco.__version__)
print('Bundled plugin dir:', mujoco.PLUGINS_DIR)
print('Demo path:', REPO_DIR / 'python/examples/electrical/demo_visualizer.py')
print('Electrical sim class:', SingleEnvSimulation)


In [ ]:
import os
import subprocess
import time
from google.colab import output

os.environ['DISPLAY'] = DISPLAY_ID
os.environ['LIBGL_ALWAYS_SOFTWARE'] = '1'

_bg_processes = globals().get('_bg_processes', {})

def start_once(name, cmd, env=None):
    proc = _bg_processes.get(name)
    if proc is not None and proc.poll() is None:
        print(f'{name} already running (pid={proc.pid})')
        return proc
    proc = subprocess.Popen(cmd, env=env or os.environ.copy(), stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    _bg_processes[name] = proc
    print(f'started {name} (pid={proc.pid})')
    return proc

start_once('xvfb', ['Xvfb', DISPLAY_ID, '-screen', '0', '1440x900x24', '-ac', '+extension', 'GLX', '+render'])
time.sleep(2)
start_once('fluxbox', ['fluxbox'], env={**os.environ, 'DISPLAY': DISPLAY_ID})
start_once('x11vnc', ['x11vnc', '-display', DISPLAY_ID, '-forever', '-shared', '-nopw', '-rfbport', '5901'])
start_once('novnc', ['websockify', '--web=/usr/share/novnc/', str(VNC_PORT), 'localhost:5901'])

print(f'Opening noVNC on port {VNC_PORT}...')
output.serve_kernel_port_as_iframe(VNC_PORT, path='/vnc.html?autoconnect=true&resize=scale', height=720)

In [ ]:
import os
import subprocess

env = os.environ.copy()
env['DISPLAY'] = DISPLAY_ID
env['LIBGL_ALWAYS_SOFTWARE'] = '1'

demo_cmd = [
    'python3',
    str(REPO_DIR / 'python/examples/electrical/demo_visualizer.py'),
    '--light',
    '--steps',
    '5000',
]

demo_proc = subprocess.Popen(demo_cmd, cwd=str(REPO_DIR), env=env)
print(f'Visualizer started with PID {demo_proc.pid}.')
print('Open the embedded noVNC pane above, then press F4 inside the viewer for the sensor panel.')


## Optional helpers

- The setup cell automatically chooses the `cp311` or `cp312` wheel based on the Colab runtime.
- To use a newer published wheel later, update `RELEASE_TAG` in the second cell.
- Re-run the **noVNC** cell if the browser frame disconnects.
- Change `--light` to `--heavy` in the launch cell to run the heavier payload scenario.
- If you want to stop the current viewer, run:

```python
import os, signal
os.kill(demo_proc.pid, signal.SIGTERM)
```
